# 02 — Grain, Candidate Keys & Duplicates

**Objetivo:** probar hipótesis de granularidad y claves sin asumir que una columna es PK solo porque se llame ID.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué representa una fila en cada fuente?
- ¿Qué claves candidatas son realmente únicas?
- ¿Cuántos nulls tienen?
- ¿Cuál es la multiplicidad máxima?

**Interpretación:** una clave no es aceptada por nombre; es aceptada por evidencia + semántica de negocio.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.profiling.keys import analyze_all_candidate_keys
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]
key_analysis = analyze_all_candidate_keys(
    FILES,
    DATASETS,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    key_analysis,
    paths.reports / "profiling" / "key_analysis.parquet",
)

key_analysis

In [ ]:
# Candidatos que podrían ser PK técnica cuando son no-null.
key_analysis.loc[
    key_analysis.get("is_unique_when_nonnull", False).fillna(False),
    ["dataset", "candidate_key", "row_count", "null_key_rate", "uniqueness_ratio", "max_multiplicity"]
]

## Trabajo manual requerido
Completa `docs/data-dictionary/*.md` con una frase explícita del grain, por ejemplo:
`1 fila = 1 evento de modificación contractual`.

Si el análisis contradice la documentación fuente, conserva ambas evidencias y no corrijas silenciosamente.